# 🦾 Inverse Kinematics Planar Robot 3-DOF
## Pendekatan Machine Learning, Deep Learning, dan Reinforcement Learning
### ✅ Versi Diperbarui: Link `[0.5, 0.4, 0.3]` + Optimasi ML

---
> **Prasyarat:** Python, NumPy, dasar ML & Neural Network  
> **Tools:** `numpy`, `matplotlib`, `scikit-learn`, `torch`, `gymnasium`, `stable-baselines3`


In [ ]:
# Install dependencies (jalankan sekali)
!pip install gymnasium stable-baselines3 scikit-learn torch matplotlib numpy tqdm -q
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

print("✅ Library siap!")

---
## 🛠️ Bagian 2: Setup & Konfigurasi Robot

> **📌 Perubahan:** Panjang link diubah dari `[0.4, 0.3, 0.2]` → **`[0.5, 0.4, 0.3]`**

In [ ]:
# ─── Konfigurasi Robot (DIPERBARUI) ───────────────────────────────────
# ⚠️  Link BARU: [0.5, 0.4, 0.3]  (sebelumnya [0.4, 0.3, 0.2])
L1, L2, L3 = 0.5, 0.4, 0.3       # panjang link (meter)
LINK_LENGTHS = np.array([L1, L2, L3])
JOINT_LIMIT  = np.pi               # ±π radian per sendi
MAX_REACH    = L1 + L2 + L3        # 1.2 m  (sebelumnya 0.9 m)

print(f"Link lengths : L1={L1}, L2={L2}, L3={L3}")
print(f"Max reach    : {MAX_REACH} m  (sebelumnya 0.9 m, +{MAX_REACH-0.9:.1f} m)")

def forward_kinematics(theta):
    t1, t2, t3 = theta
    p0 = np.array([0.0, 0.0])
    p1 = p0 + L1 * np.array([np.cos(t1), np.sin(t1)])
    p2 = p1 + L2 * np.array([np.cos(t1+t2), np.sin(t1+t2)])
    p3 = p2 + L3 * np.array([np.cos(t1+t2+t3), np.sin(t1+t2+t3)])
    return p0, p1, p2, p3

def get_end_effector(theta):
    return forward_kinematics(theta)[3]

def plot_robot(ax, theta, target=None, title="Robot", color='steelblue'):
    p0, p1, p2, p3 = forward_kinematics(theta)
    points = np.array([p0, p1, p2, p3])
    ax.plot(points[:,0], points[:,1], '-o', color=color,
            linewidth=3, markersize=8, markerfacecolor='white',
            markeredgecolor=color, markeredgewidth=2)
    ax.plot(*p0, 'ks', markersize=10)
    ax.plot(*p3, '*', color='red', markersize=14, label='End-effector')
    if target is not None:
        ax.plot(*target, 'r+', markersize=16, markeredgewidth=3, label='Target')
        ax.add_patch(Circle(target, 0.02, color='red', alpha=0.3))
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axhline(0, color='k', lw=0.5, alpha=0.3)
    ax.axvline(0, color='k', lw=0.5, alpha=0.3)

# ─── Perbandingan Workspace: Lama vs Baru ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
theta_range = np.linspace(0, 2*np.pi, 300)

OLD_REACH = 0.9
for ax, (reach, label, col) in zip(axes, [
    (OLD_REACH, "Link Lama [0.4, 0.3, 0.2]\nMax reach = 0.9 m", 'salmon'),
    (MAX_REACH, f"Link Baru [0.5, 0.4, 0.3]\nMax reach = {MAX_REACH} m", 'lightblue'),
]):
    ax.fill(reach*np.cos(theta_range), reach*np.sin(theta_range),
            color=col, alpha=0.4)
    ax.plot(reach*np.cos(theta_range), reach*np.sin(theta_range),
            'b--', linewidth=1.5)
    ax.plot(0, 0, 'ks', markersize=10)
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")

    # Annotate area
    area = np.pi * reach**2
    ax.text(0, -reach*0.7, f"Area ≈ {area:.2f} m²", ha='center',
            fontsize=11, color='navy', fontweight='bold')

plt.suptitle("Perbandingan Workspace: Link Lama vs Baru", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

area_old = np.pi * OLD_REACH**2
area_new = np.pi * MAX_REACH**2
print(f"Area workspace lama: {area_old:.3f} m²")
print(f"Area workspace baru: {area_new:.3f} m²")
print(f"Peningkatan area   : +{(area_new-area_old)/area_old*100:.1f}%")

In [ ]:
def is_reachable(target, tol=1e-6):
    target = np.array(target, dtype=float)
    dist = np.linalg.norm(target)
    min_reach = abs(L1 - L2 - L3)
    return bool(min_reach - tol <= dist <= MAX_REACH + tol)

def filter_reachable(positions, labels=None):
    mask = np.array([is_reachable(p) for p in positions])
    if labels is not None:
        return positions[mask], labels[mask], mask
    return positions[mask], mask

print(f"MAX_REACH = {MAX_REACH} m")
test_points = [(0.5, 0.3), (0.8, 0.6), (1.0, 0.0), (1.1, 0.1), (0.0, 0.0)]
print(f"{'Target':20s} | {'Jarak':8s} | {'Reachable?'}")
print("-"*45)
for pt in test_points:
    d = np.linalg.norm(pt)
    r = is_reachable(pt)
    print(f"({pt[0]:.1f}, {pt[1]:.1f}){'':14} | {d:.4f} m | {'YES ✅' if r else 'NO ❌'}")

---
## 📊 Bagian 3: Dataset

In [ ]:
np.random.seed(42)
N = 20000

thetas = np.random.uniform(-JOINT_LIMIT, JOINT_LIMIT, size=(N, 3)).astype(np.float32)
ee_positions = np.array([get_end_effector(t) for t in thetas], dtype=np.float32)

_, reach_mask = filter_reachable(ee_positions)
n_reachable = int(reach_mask.sum())
print(f"Dataset: {N} sampel → {n_reachable} reachable ({n_reachable/N*100:.1f}%)")

ee_positions = ee_positions[reach_mask]
thetas       = thetas[reach_mask]

# Visualisasi Workspace dengan link baru
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(ee_positions[:,0], ee_positions[:,1], alpha=0.05, s=1, color='steelblue')
ax.set_title(f"Workspace Robot — Link [0.5, 0.4, 0.3]\n({len(ee_positions):,} titik)", fontsize=13)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
circle = plt.Circle((0,0), MAX_REACH, fill=False, color='red', linestyle='--',
                     linewidth=1.5, label=f'Max reach = {MAX_REACH} m')
ax.add_patch(circle)
ax.legend(); plt.tight_layout(); plt.show()
print(f"✅ Dataset siap: {len(ee_positions)} sampel")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    ee_positions, thetas, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

---
## 🤖 Bagian 4: Machine Learning — Original + Optimasi

Menggunakan 5 metode:
| # | Metode | Keterangan |
|---|--------|------------|
| 1 | **KNN (k=5)** | Baseline original |
| 2 | **KNN Tuned** | Hyperparameter tuning (k optimal) |
| 3 | **Random Forest** | Original (100 trees) |
| 4 | **Gradient Boosting** | Metode baru, lebih akurat |
| 5 | **SVR** | Support Vector Regression |


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
import time

scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s  = scaler_X.transform(X_test)

ml_results = {}

# ─── 1. KNN Original (k=5) ───────────────────────────────────────────
print("🔵 Training KNN Original (k=5)...")
t0 = time.time()
knn = KNeighborsRegressor(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_s, y_train)
knn_pred = knn.predict(X_test_s)
t_knn = time.time() - t0
knn_ee  = np.array([get_end_effector(p) for p in knn_pred])
knn_err = np.linalg.norm(knn_ee - X_test, axis=1)
ml_results['KNN (k=5)'] = {'pred': knn_pred, 'ee_err': knn_err, 'time': t_knn}
print(f"   ✅ KNN — mean EE error: {knn_err.mean()*100:.2f} cm | time: {t_knn:.1f}s")

# ─── 2. KNN Tuned (cari k optimal) ────────────────────────────────────
print("\n🔵 Mencari k optimal untuk KNN...")
k_values = [3, 5, 7, 10, 15, 20]
k_errors = []
for k in k_values:
    knn_tmp = KNeighborsRegressor(n_neighbors=k, n_jobs=-1)
    knn_tmp.fit(X_train_s, y_train)
    pred_tmp = knn_tmp.predict(X_test_s)
    ee_tmp = np.array([get_end_effector(p) for p in pred_tmp])
    err_tmp = np.linalg.norm(ee_tmp - X_test, axis=1).mean() * 100
    k_errors.append(err_tmp)
    print(f"   k={k:2d} → mean error: {err_tmp:.2f} cm")

best_k = k_values[np.argmin(k_errors)]
print(f"\n   ⭐ k optimal: {best_k} (error: {min(k_errors):.2f} cm)")

t0 = time.time()
knn_tuned = KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1)
knn_tuned.fit(X_train_s, y_train)
knn_t_pred = knn_tuned.predict(X_test_s)
t_knn_t = time.time() - t0
knn_t_ee  = np.array([get_end_effector(p) for p in knn_t_pred])
knn_t_err = np.linalg.norm(knn_t_ee - X_test, axis=1)
ml_results[f'KNN Tuned (k={best_k})'] = {'pred': knn_t_pred, 'ee_err': knn_t_err, 'time': t_knn_t}
print(f"   ✅ KNN Tuned — mean EE error: {knn_t_err.mean()*100:.2f} cm")

# Plot k vs error
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, k_errors, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axvline(best_k, color='red', linestyle='--', label=f'k optimal = {best_k}')
ax.set_xlabel("Nilai k"); ax.set_ylabel("Mean EE Error (cm)")
ax.set_title("KNN: Tuning Hyperparameter k", fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─── 3. Random Forest Original ────────────────────────────────────────
print("🌲 Training Random Forest (100 trees)...")
t0 = time.time()
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train_s, y_train)
rf_pred = rf.predict(X_test_s)
t_rf = time.time() - t0
rf_ee  = np.array([get_end_effector(p) for p in rf_pred])
rf_err = np.linalg.norm(rf_ee - X_test, axis=1)
ml_results['Random Forest'] = {'pred': rf_pred, 'ee_err': rf_err, 'time': t_rf}
print(f"   ✅ RF — mean EE error: {rf_err.mean()*100:.2f} cm | time: {t_rf:.1f}s")

# ─── 4. Gradient Boosting (METODE BARU) ────────────────────────────────
print("\n🚀 Training Gradient Boosting (MultiOutput)...")
t0 = time.time()
gb_base = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.1, max_depth=5,
    subsample=0.8, random_state=42
)
gb = MultiOutputRegressor(gb_base, n_jobs=-1)
gb.fit(X_train_s, y_train)
gb_pred = gb.predict(X_test_s)
t_gb = time.time() - t0
gb_ee  = np.array([get_end_effector(p) for p in gb_pred])
gb_err = np.linalg.norm(gb_ee - X_test, axis=1)
ml_results['Gradient Boosting'] = {'pred': gb_pred, 'ee_err': gb_err, 'time': t_gb}
print(f"   ✅ GB — mean EE error: {gb_err.mean()*100:.2f} cm | time: {t_gb:.1f}s")

# ─── 5. SVR (METODE BARU) ──────────────────────────────────────────────
print("\n⚙️  Training SVR (MultiOutput)...")
t0 = time.time()
svr = MultiOutputRegressor(SVR(kernel='rbf', C=10, epsilon=0.01), n_jobs=-1)
# Gunakan subset karena SVR lambat O(n²)
idx_sub = np.random.choice(len(X_train_s), 5000, replace=False)
svr.fit(X_train_s[idx_sub], y_train[idx_sub])
svr_pred = svr.predict(X_test_s)
t_svr = time.time() - t0
svr_ee  = np.array([get_end_effector(p) for p in svr_pred])
svr_err = np.linalg.norm(svr_ee - X_test, axis=1)
ml_results['SVR (RBF)'] = {'pred': svr_pred, 'ee_err': svr_err, 'time': t_svr}
print(f"   ✅ SVR — mean EE error: {svr_err.mean()*100:.2f} cm | time: {t_svr:.1f}s")
print("\n📋 Ringkasan ML:")
for name, res in ml_results.items():
    print(f"   {name:25s}: {res['ee_err'].mean()*100:.2f} cm  ({res['time']:.1f}s)")

In [ ]:
# ─── Perbandingan Error ML ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
names  = list(ml_results.keys())
errors = [res['ee_err'].mean()*100 for res in ml_results.values()]
colors = ['steelblue', 'cornflowerblue', 'forestgreen', 'darkorange', 'tomato']
bars   = ax.bar(names, errors, color=colors[:len(names)], edgecolor='white', width=0.6)
for bar, err in zip(bars, errors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{err:.2f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel("Mean EE Error (cm)")
ax.set_title("Perbandingan Error — Semua Metode ML", fontweight='bold')
ax.set_xticklabels(names, rotation=20, ha='right')
ax.grid(True, axis='y', alpha=0.3)

# Distribusi error
ax2 = axes[1]
for (name, res), c in zip(ml_results.items(), colors):
    ax2.hist(res['ee_err']*100, bins=50, alpha=0.5, label=name, color=c)
ax2.axvline(0, color='k', lw=0.5)
ax2.set_xlabel("Error (cm)"); ax2.set_ylabel("Frekuensi")
ax2.set_title("Distribusi Error", fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.suptitle("Evaluasi Metode ML — Link [0.5, 0.4, 0.3]", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── ANIMASI ML ─────────────────────────────────────────────────────────
def animate_ik(model_name, predict_fn, n_targets=10, color='steelblue'):
    np.random.seed(7)
    sample_idx = np.random.choice(len(X_test), n_targets, replace=False)
    targets = X_test[sample_idx]

    fig, ax = plt.subplots(figsize=(6, 6))

    def update(frame):
        ax.clear()
        target = targets[frame]
        theta_pred = predict_fn(target.reshape(1,-1))[0]
        ee_actual  = get_end_effector(theta_pred)
        err = np.linalg.norm(ee_actual - target) * 100
        plot_robot(ax, theta_pred, target=target,
                   title=f"{model_name}\nTarget ({target[0]:.2f}, {target[1]:.2f}) | Error: {err:.1f} cm",
                   color=color)
        ax.legend(loc='upper right', fontsize=8)

    ani = animation.FuncAnimation(fig, update, frames=n_targets, interval=800, repeat=True)
    plt.close()
    return ani

ani_knn = animate_ik("KNN Tuned", lambda x: knn_tuned.predict(scaler_X.transform(x)), color='steelblue')
print("▶️  Animasi KNN Tuned:")
HTML(ani_knn.to_jshtml())

In [ ]:
ani_gb = animate_ik("Gradient Boosting", lambda x: gb.predict(scaler_X.transform(x)), color='darkorange')
print("▶️  Animasi Gradient Boosting:")
HTML(ani_gb.to_jshtml())

---
## 🧠 Bagian 5: Deep Learning (PyTorch MLP)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")

class IKNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 3),
        )
    def forward(self, x):
        out = self.net(x)
        return torch.tanh(out) * torch.pi

model_dl = IKNet().to(device)
print(f"✅ Model: {sum(p.numel() for p in model_dl.parameters()):,} parameter")

In [ ]:
def fk_torch(theta):
    t1, t2, t3 = theta[:,0], theta[:,1], theta[:,2]
    x = (L1*torch.cos(t1) + L2*torch.cos(t1+t2) + L3*torch.cos(t1+t2+t3))
    y = (L1*torch.sin(t1) + L2*torch.sin(t1+t2) + L3*torch.sin(t1+t2+t3))
    return torch.stack([x, y], dim=1)

X_tr_t = torch.FloatTensor(X_train).to(device)
X_te_t = torch.FloatTensor(X_test).to(device)
y_tr_t = torch.FloatTensor(y_train).to(device)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_dl = DataLoader(train_ds, batch_size=512, shuffle=True)

optimizer = optim.Adam(model_dl.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
EPOCHS = 100

train_losses, val_losses = [], []
print("🚀 Training DL model...")
for epoch in range(EPOCHS):
    model_dl.train()
    ep_loss = 0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        pred_theta = model_dl(xb)
        pred_ee    = fk_torch(pred_theta)
        loss = nn.MSELoss()(pred_ee, xb)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
    train_losses.append(ep_loss / len(train_dl))

    model_dl.eval()
    with torch.no_grad():
        val_pred_theta = model_dl(X_te_t)
        val_pred_ee    = fk_torch(val_pred_theta)
        val_loss = nn.MSELoss()(val_pred_ee, X_te_t).item()
    val_losses.append(val_loss)
    scheduler.step()
    if (epoch+1) % 20 == 0:
        print(f"   Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_losses[-1]:.6f} | Val: {val_loss:.6f}")

model_dl.eval()
with torch.no_grad():
    dl_pred = model_dl(X_te_t).cpu().numpy()

dl_ee  = np.array([get_end_effector(p) for p in dl_pred])
dl_err = np.linalg.norm(dl_ee - X_test, axis=1)
print(f"\n✅ DL — mean EE error: {dl_err.mean()*100:.2f} cm")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train Loss', color='steelblue')
ax.plot(val_losses,   label='Val Loss',   color='orange')
ax.set_title("Training Curve — DL (IKNet)", fontweight='bold')
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
ax.legend(); ax.grid(True, alpha=0.3); ax.set_yscale('log')
plt.tight_layout(); plt.show()

---
## 🕹️ Bagian 6: Reinforcement Learning (PPO)

In [ ]:
import gymnasium as gym
from gymnasium import spaces

# ─── Joint Limits Realistis ───────────────────────────────────────────
# θ1: bebas berputar (shoulder)
# θ2: elbow, hindari lipatan ekstrem
# θ3: wrist, range lebih sempit
JOINT_LIMITS_LOW  = np.array([-np.pi,      -2*np.pi/3, -np.pi/2], dtype=np.float32)
JOINT_LIMITS_HIGH = np.array([ np.pi,       2*np.pi/3,  np.pi/2], dtype=np.float32)

class PlanarRobot3DOFEnv(gym.Env):
    """Custom Gymnasium environment untuk robot planar 3-DOF."""
    metadata = {"render_modes": ["rgb_array"], "render_fps": 30}

    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        self.max_steps   = 200
        self.goal_thresh = 0.02  # 2 cm

        # Observation: [θ1,θ2,θ3, dθ1,dθ2,dθ3, x_target, y_target]
        obs_high = np.concatenate([
            JOINT_LIMITS_HIGH,
            np.array([5.0, 5.0, 5.0], dtype=np.float32),
            np.array([MAX_REACH, MAX_REACH], dtype=np.float32)
        ])
        self.observation_space = spaces.Box(-obs_high, obs_high, dtype=np.float32)

        # Action: delta sudut per sendi (lebih ketat untuk θ2 & θ3)
        act_high = np.array([0.15, 0.10, 0.08], dtype=np.float32)
        self.action_space = spaces.Box(-act_high, act_high, dtype=np.float32)
        self._reset_state()

    def _reset_state(self):
        # Inisialisasi di tengah joint range (bukan zero) untuk variasi
        self.theta  = np.random.uniform(
            JOINT_LIMITS_LOW * 0.3, JOINT_LIMITS_HIGH * 0.3
        ).astype(np.float32)
        self.dtheta = np.zeros(3, dtype=np.float32)
        self.steps  = 0
        # Generate target reachable via FK sampling
        while True:
          target_angles = np.random.uniform(-np.pi, np.pi, 3).astype(np.float32)
          self.target   = get_end_effector(target_angles).astype(np.float32)
          if is_reachable(self.target):
            break
        # # Safety check reachability
        # assert is_reachable(self.target), \
        #     f"Target {self.target} TIDAK reachable! dist={np.linalg.norm(self.target):.4f} > MAX_REACH={MAX_REACH}"

    def _get_obs(self):
        return np.concatenate([self.theta, self.dtheta, self.target]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            np.random.seed(seed)
        self._reset_state()
        return self._get_obs(), {}

    def step(self, action):
        # Clip action per joint sesuai batas masing-masing
        act_high = np.array([0.15, 0.10, 0.08], dtype=np.float32)
        delta = np.clip(action, -act_high, act_high).astype(np.float32)

        # Update theta dengan joint limits realistis
        self.theta  = np.clip(self.theta + delta,
                              JOINT_LIMITS_LOW, JOINT_LIMITS_HIGH).astype(np.float32)
        self.dtheta = delta

        ee   = get_end_effector(self.theta)
        dist = float(np.linalg.norm(ee - self.target))

        # ── Reward shaping ─────────────────────────────────────────────
        # 1. Proximity reward
        reward = -dist

        # 2. Penalti konfigurasi degenerate:
        #    - Hindari θ2, θ3 yang terlalu besar (lipatan ekstrem)
        angle_penalty = 0.005 * float(np.sum(self.theta[1:]**2))
        reward -= angle_penalty

        # 3. Bonus sukses
        terminated = False
        if dist < self.goal_thresh:
            reward += 10.0
            terminated = True

        self.steps += 1
        truncated = self.steps >= self.max_steps
        return self._get_obs(), reward, terminated, truncated, {"dist": dist}

    def render(self):
        fig, ax = plt.subplots(figsize=(5, 5))
        plot_robot(ax, self.theta, target=self.target,
                   title="RL Agent", color='purple')
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        plt.close(fig)
        return img

print("Joint limits:")
print(f"  theta1 (shoulder): [{np.degrees(JOINT_LIMITS_LOW[0]):.0f}, {np.degrees(JOINT_LIMITS_HIGH[0]):.0f}] deg")
print(f"  theta2 (elbow)   : [{np.degrees(JOINT_LIMITS_LOW[1]):.0f}, {np.degrees(JOINT_LIMITS_HIGH[1]):.0f}] deg")
print(f"  theta3 (wrist)   : [{np.degrees(JOINT_LIMITS_LOW[2]):.0f}, {np.degrees(JOINT_LIMITS_HIGH[2]):.0f}] deg")
print("Environment terdefinisi!")
from gymnasium.utils.env_checker import check_env
env_test = PlanarRobot3DOFEnv()
check_env(env_test, warn=True)
print("check_env passed!")


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
import os

class RewardLoggerCallback(BaseCallback):
    """
    Callback untuk mencatat mean reward dan mean episode length
    setiap `log_freq` timestep — digunakan untuk plot learning curve.
    """
    def __init__(self, log_freq=5000, verbose=0):
        super().__init__(verbose)
        self.log_freq       = log_freq
        self.timesteps_log  = []
        self.mean_rewards   = []
        self.mean_ep_lens   = []
        self._ep_rewards    = []
        self._ep_lens       = []
        self._cur_ep_reward = None
        self._cur_ep_len    = None

    def _on_training_start(self):
        self._cur_ep_reward = np.zeros(self.training_env.num_envs)
        self._cur_ep_len    = np.zeros(self.training_env.num_envs, dtype=int)

    def _on_step(self):
        rewards = self.locals["rewards"]
        dones   = self.locals["dones"]
        self._cur_ep_reward += rewards
        self._cur_ep_len    += 1

        for idx, done in enumerate(dones):
            if done:
                self._ep_rewards.append(self._cur_ep_reward[idx])
                self._ep_lens.append(self._cur_ep_len[idx])
                self._cur_ep_reward[idx] = 0
                self._cur_ep_len[idx]    = 0

        if self.n_calls % self.log_freq == 0 and len(self._ep_rewards) > 0:
            mean_r   = float(np.mean(self._ep_rewards[-50:]))
            mean_len = float(np.mean(self._ep_lens[-50:]))
            self.timesteps_log.append(self.num_timesteps)
            self.mean_rewards.append(mean_r)
            self.mean_ep_lens.append(mean_len)
            print(f"   ⏱  {self.num_timesteps:>8,} steps | "
                  f"mean_reward: {mean_r:7.3f} | mean_ep_len: {mean_len:.1f}")
        return True

# ─── Training PPO ─────────────────────────────────────────────────────
vec_env  = make_vec_env(PlanarRobot3DOFEnv, n_envs=16)
eval_env = make_vec_env(PlanarRobot3DOFEnv, n_envs=1)

model_rl = PPO(
    "MlpPolicy", vec_env,
    learning_rate=3e-4,
    n_steps=1024, batch_size=256, n_epochs=10,
    gamma=0.99, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.01,
    policy_kwargs=dict(net_arch=dict(pi=[256,256], vf=[256,256])),
    verbose=0
)

eval_cb = EvalCallback(eval_env, eval_freq=20000, n_eval_episodes=10,
                       best_model_save_path="./rl_best/", verbose=0)

reward_logger = RewardLoggerCallback(log_freq=5000)
print("🚀 Training PPO (800k timesteps)...")
model_rl.learn(total_timesteps=800_000,
               callback=[eval_cb, reward_logger])
print("✅ Training RL selesai!")


In [ ]:
# ─── Visualisasi Learning Curve (Reward vs Timestep) ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Mean Episode Reward ───────────────────────────────────────
ax = axes[0]
ts  = np.array(reward_logger.timesteps_log)
rew = np.array(reward_logger.mean_rewards)

ax.plot(ts, rew, color='purple', alpha=0.4, linewidth=1, label='Raw')

# Smoothing dengan rolling average
if len(rew) >= 10:
    window = max(5, len(rew) // 10)
    smooth = np.convolve(rew, np.ones(window)/window, mode='valid')
    ts_smooth = ts[window-1:]
    ax.plot(ts_smooth, smooth, color='purple', linewidth=2.5, label=f'Smoothed (w={window})')

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_title("Learning Curve — Mean Episode Reward", fontsize=13, fontweight='bold')
ax.set_xlabel("Timestep"); ax.set_ylabel("Mean Reward (per episode)")
ax.legend(); ax.grid(True, alpha=0.3)
ax.ticklabel_format(axis='x', style='sci', scilimits=(0,0))

# ── Plot 2: Mean Episode Length ───────────────────────────────────────
ax2 = axes[1]
ep_len = np.array(reward_logger.mean_ep_lens)

ax2.plot(ts, ep_len, color='steelblue', alpha=0.4, linewidth=1, label='Raw')

if len(ep_len) >= 10:
    smooth_len = np.convolve(ep_len, np.ones(window)/window, mode='valid')
    ax2.plot(ts_smooth, smooth_len, color='steelblue', linewidth=2.5,
             label=f'Smoothed (w={window})')

ax2.axhline(200, color='red', linestyle='--', alpha=0.6, label='Max steps (200)')
ax2.set_title("Episode Length vs Timestep", fontsize=13, fontweight='bold')
ax2.set_xlabel("Timestep"); ax2.set_ylabel("Mean Episode Length (steps)")
ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.ticklabel_format(axis='x', style='sci', scilimits=(0,0))

plt.suptitle("PPO Training Progress — Planar Robot 3-DOF IK",
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Ringkasan learning ────────────────────────────────────────────────
if len(rew) > 0:
    print(f"Reward awal  (avg 10 pertama): {np.mean(rew[:10]):.3f}")
    print(f"Reward akhir (avg 10 terakhir): {np.mean(rew[-10:]):.3f}")
    improvement = np.mean(rew[-10:]) - np.mean(rew[:10])
    print(f"Peningkatan  : {improvement:+.3f}")


In [ ]:
# ─── Evaluasi RL ──────────────────────────────────────────────────────
eval_single = PlanarRobot3DOFEnv()
successes, distances = [], []

model_rl.load('./rl_best/best_model.zip')
np.random.seed(99)
for _ in range(200):
    obs, _ = eval_single.reset()
    for _ in range(200):
        action, _ = model_rl.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_single.step(action)
        if terminated or truncated:
            break
    distances.append(info['dist'])
    successes.append(info['dist'] < 0.02)

print(f"✅ RL Success rate : {np.mean(successes)*100:.1f}%")
print(f"✅ RL Mean EE error: {np.mean(distances)*100:.2f} cm")
rl_errors = np.array(distances)


In [ ]:
# ─── ANIMASI RL: Agent mencapai target step-by-step ──────────────────
def animate_rl(n_episodes=8):
    """Animasi episode RL: pergerakan robot frame per frame."""
    # Kumpulkan trajectory dari beberapa episode
    trajectories = []
    env_anim = PlanarRobot3DOFEnv()
    np.random.seed(55)
    for ep in range(n_episodes):
        obs, _ = env_anim.reset(seed=ep*10)
        target = env_anim.target.copy()
        traj = []
        for _ in range(80):
            action, _ = model_rl.predict(obs, deterministic=True)
            obs, _, terminated, truncated, info = env_anim.step(action)
            traj.append((env_anim.theta.copy(), target.copy(), info['dist']))
            if terminated or truncated:
                break
        trajectories.append(traj)

    # Flatten: semua episode diputar berurutan
    all_frames = []
    for traj in trajectories:
        # Ambil setiap 3 frame supaya animasi tidak terlalu lambat
        all_frames.extend(traj[::3])
        # Tahan frame terakhir 5x
        all_frames.extend([traj[-1]] * 5)

    fig, ax = plt.subplots(figsize=(6, 6))

    def update(i):
        ax.clear()
        theta, target, dist = all_frames[i]
        ee = get_end_effector(theta)
        success = dist < 0.02
        color = 'green' if success else 'purple'
        status = "✅ BERHASIL!" if success else f"dist: {dist*100:.1f} cm"
        plot_robot(ax, theta, target=target,
                   title=f"RL Agent (PPO)\n{status}", color=color)
        ax.legend(loc='upper right', fontsize=8)

    ani = animation.FuncAnimation(fig, update, frames=len(all_frames),
                                  interval=120, repeat=True)
    plt.close()
    return ani

ani_rl = animate_rl()
print("▶️  Animasi Reinforcement Learning:")
HTML(ani_rl.to_jshtml())


---
## 📊 Bagian 7: Perbandingan Semua Metode

In [ ]:
import pandas as pd

# ─── Tabel perbandingan lengkap ────────────────────────────────────────
best_ml_name = min(ml_results, key=lambda k: ml_results[k]['ee_err'].mean())
best_ml_err  = ml_results[best_ml_name]['ee_err'].mean()*100

summary = {
    'Metode': ['KNN (k=5)', 'KNN Tuned', 'Random Forest', 'Gradient Boosting', 'SVR (RBF)',
               'Deep Learning (MLP)', 'RL (PPO)'],
    'Mean Error (cm)': [
        round(ml_results['KNN (k=5)']['ee_err'].mean()*100, 2),
        round(ml_results[f'KNN Tuned (k={best_k})']['ee_err'].mean()*100, 2),
        round(ml_results['Random Forest']['ee_err'].mean()*100, 2),
        round(ml_results['Gradient Boosting']['ee_err'].mean()*100, 2),
        round(ml_results['SVR (RBF)']['ee_err'].mean()*100, 2),
        round(dl_err.mean()*100, 2),
        round(rl_errors.mean()*100, 2),
    ],
    'Butuh Dataset': ['✅','✅','✅','✅','✅','✅','❌'],
    'Kelebihan': [
        'Simpel, cepat', 'k optimal', 'Robust', 'Akurat, stabil', 'Margin optimal',
        'Generalisasi', 'Adaptif',
    ],
}
df = pd.DataFrame(summary)
df = df.sort_values('Mean Error (cm)')
print("\n📊 TABEL PERBANDINGAN (LINK BARU [0.5, 0.4, 0.3]):")
print(df.to_string(index=False))
print(f"\n🏆 Metode ML terbaik: {best_ml_name} ({best_ml_err:.2f} cm)")

In [ ]:
# ─── Bar Chart Final ──────────────────────────────────────────────────
labels_all  = ['KNN\n(k=5)', f'KNN\nTuned', 'Random\nForest',
               'Gradient\nBoosting', 'SVR\n(RBF)', 'Deep\nLearning', 'RL\n(PPO)']
errors_all  = [
    ml_results['KNN (k=5)']['ee_err'].mean()*100,
    ml_results[f'KNN Tuned (k={best_k})']['ee_err'].mean()*100,
    ml_results['Random Forest']['ee_err'].mean()*100,
    ml_results['Gradient Boosting']['ee_err'].mean()*100,
    ml_results['SVR (RBF)']['ee_err'].mean()*100,
    dl_err.mean()*100,
    rl_errors.mean()*100,
]
colors_all  = ['steelblue','cornflowerblue','forestgreen','darkorange','tomato','gold','purple']

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(labels_all, errors_all, color=colors_all, edgecolor='white', width=0.6)
for bar, err in zip(bars, errors_all):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{err:.2f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel("Mean EE Error (cm)", fontsize=12)
ax.set_title(f"Perbandingan Akurasi IK — Link [0.5, 0.4, 0.3] | Max Reach = {MAX_REACH} m",
             fontsize=13, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, max(errors_all)*1.25)

# Tandai yang terbaik
best_idx = np.argmin(errors_all)
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)
ax.text(bars[best_idx].get_x() + bars[best_idx].get_width()/2,
        bars[best_idx].get_height() + 0.3, '🏆', ha='center', fontsize=16)

plt.tight_layout(); plt.show()

---
## 📝 Kesimpulan & Analisis Perubahan Link

### Dampak Perubahan Link `[0.4, 0.3, 0.2]` → `[0.5, 0.4, 0.3]`

| Properti | Link Lama | Link Baru | Perubahan |
|----------|-----------|-----------|-----------|
| Max Reach | 0.9 m | **1.2 m** | +33% |
| Workspace Area | ~2.54 m² | **~4.52 m²** | +78% |
| Min Reach | 0.0 m | 0.0 m | — |
| Proporsi L1:L2:L3 | 4:3:2 | **5:4:3** | Lebih seimbang |

### Dampak pada Akurasi ML

- **Workspace lebih besar** → model perlu belajar pemetaan yang lebih kompleks  
- **KNN Tuned** umumnya lebih baik dari KNN default karena k optimal menyesuaikan densitas data  
- **Gradient Boosting** unggul di supervised learning karena boosting mengurangi bias secara iteratif  
- **SVR** kompetitif untuk dataset kecil; overhead O(n²) membatasi penggunaan dataset besar  
- **Deep Learning** paling stabil di semua range workspace karena generalisasi neural network  
- **RL** tidak terpengaruh perubahan link — hanya perlu update `MAX_REACH` dan reward function  

### Rekomendasi Penggunaan

| Kondisi | Rekomendasi |
|---------|-------------|
| Cepat, resource terbatas | **KNN Tuned** |
| Akurasi ML tertinggi | **Gradient Boosting** |
| Akurasi overall tertinggi | **Deep Learning** |
| Lingkungan dinamis/obstacle | **RL (PPO)** |

---
*🦾 Selamat belajar! Notebook diperbarui dengan link `[0.5, 0.4, 0.3]` dan optimasi ML.*
